In [16]:
# ========================================
# ✈️ Smart Tourism Guide Chatbot (FAISS + NVIDIA RAG + Gradio)
# ========================================

# Install dependencies
!pip install -q gradio faiss-cpu sentence-transformers requests langchain pypdf2 python-docx pandas

import os
import gradio as gr
import faiss
import requests
import pandas as pd
from sentence_transformers import SentenceTransformer
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document
from PyPDF2 import PdfReader
import docx

# NVIDIA API
NVIDIA_API_KEY = "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"  # Replace with your NVIDIA API key
NVIDIA_CHAT_URL = "https://integrate.api.nvidia.com/v1/chat/completions"
headers = {"Authorization": f"Bearer {NVIDIA_API_KEY}", "Content-Type": "application/json"}

# Embeddings
embedder = SentenceTransformer("all-MiniLM-L6-v2")
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

def load_file(file_path):
    ext = file_path.split(".")[-1].lower()
    if ext == "pdf":
        reader = PdfReader(file_path)
        return "\n".join([p.extract_text() for p in reader.pages if p.extract_text()])
    elif ext == "docx":
        doc = docx.Document(file_path)
        return "\n".join([para.text for para in doc.paragraphs])
    elif ext == "csv":
        df = pd.read_csv(file_path)
        return df.astype(str).to_string()
    else:
        return open(file_path, "r", encoding="utf-8", errors="ignore").read()

def build_faiss_index(docs):
    chunks, texts = [], []
    for doc in docs:
        parts = splitter.split_text(doc)
        chunks.extend([Document(page_content=p) for p in parts])
    texts = [c.page_content for c in chunks]
    vectors = embedder.encode(texts)
    index = faiss.IndexFlatL2(vectors.shape[1])
    index.add(vectors)
    return index, texts

def nvidia_llm(query, context=""):
    payload = {
        "model": "meta/llama-3.1-8b-instruct",
        "messages": [
            {"role": "system", "content": "You are a helpful travel & tourism guide."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion:\n{query}"}
        ],
        "temperature": 0.3
    }
    resp = requests.post(NVIDIA_CHAT_URL, headers=headers, json=payload)
    return resp.json()["choices"][0]["message"]["content"]

index, texts = None, []

def rag_agent(query, history):
    global index, texts
    if index is None:
        answer = "⚠️ Please upload the tourism guide PDF first."
        history.append((query, answer))
        return history, history
    q_vec = embedder.encode([query])
    D, I = index.search(q_vec, k=3)
    context = "\n".join([texts[i] for i in I[0]])
    answer = nvidia_llm(query, context)
    history.append((query, answer))
    return history, history

def upload_and_build(file):
    global index, texts
    text = load_file(file.name)
    index, texts = build_faiss_index([text])
    return f"✅ Knowledge base built from {os.path.basename(file.name)}"

with gr.Blocks() as demo:
    gr.Markdown("## ✈️ Smart Tourism Guide Chatbot")

    with gr.Row():
        file_upload = gr.File(label="📂 Upload Tourism Guide", file_types=[".pdf", ".txt", ".docx", ".csv"])
        upload_status = gr.Textbox(label="Upload Status")

    file_upload.upload(upload_and_build, file_upload, upload_status)

    chat_ui = gr.Chatbot(label="💬 Ask about attractions, hotels, visas, travel seasons...")
    msg = gr.Textbox(label="Type your question here...")
    clear = gr.Button("Clear Chat")

    state = gr.State([])

    msg.submit(rag_agent, [msg, state], [chat_ui, state])
    clear.click(lambda: ([], []), None, [chat_ui, state])

demo.launch(inline=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 17.8 MB/s eta 0:00:00


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipython-input-3964228551.py:95: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chat_ui = gr.Chatbot(label="💬 Ask about attractions, hotels, visas, travel seasons...")


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c06430818583acbfc7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
